In [2]:
import time
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
df = pd.read_csv('StressLevelDataset.csv')

In [4]:
df.head()

,anxiety_level,self_esteem,mental_health_history,depression,headache,blood_pressure,sleep_quality,breathing_problem,noise_level,living_conditions,...,basic_needs,academic_performance,study_load,teacher_student_relationship,future_career_concerns,social_support,peer_pressure,extracurricular_activities,bullying,stress_level
0,14,20,0,11,2,1,2,4,2,3,...,2,3,2,3,3,2,3,3,2,1
1,15,8,1,15,5,3,1,4,3,1,...,2,1,4,1,5,1,4,5,5,2
2,12,18,1,14,2,1,2,2,2,2,...,2,2,3,3,2,2,3,2,2,1
3,16,12,1,15,4,3,1,3,4,2,...,2,2,4,1,4,1,4,4,5,2
4,16,28,0,7,2,3,5,1,3,2,...,3,4,3,1,2,1,5,0,5,1


In [5]:
df.shape

(1100, 21)

In [6]:
print("Mental Health:")
print(df['mental_health_history'].value_counts())

Mental Health:
mental_health_history
0    558
1    542
Name: count, dtype: int64


##  Preprocessing

In [7]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
anxiety_level                   0
self_esteem                     0
mental_health_history           0
depression                      0
headache                        0
blood_pressure                  0
sleep_quality                   0
breathing_problem               0
noise_level                     0
living_conditions               0
safety                          0
basic_needs                     0
academic_performance            0
study_load                      0
teacher_student_relationship    0
future_career_concerns          0
social_support                  0
peer_pressure                   0
extracurricular_activities      0
bullying                        0
stress_level                    0
dtype: int64


In [9]:
X = df[['anxiety_level','self_esteem','depression','headache','blood_pressure','sleep_quality','breathing_problem','noise_level','living_conditions','safety','basic_needs','academic_performance','study_load','teacher_student_relationship','future_career_concerns','social_support','peer_pressure','extracurricular_activities','bullying','stress_level']]
y = df['mental_health_history']

## Train Test split

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import pandas as pd

# Split data into training and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

# Create a DataFrame for the training set to facilitate resampling
train_df = pd.concat([X_train, y_train], axis=1)

# Separate majority and minority classes
df_majority = train_df[train_df['mental_health_history'] == 0]
df_minority = train_df[train_df['mental_health_history'] == 1]

print("Class distribution before resampling (train):")
print(train_df['mental_health_history'].value_counts())


Training set size: 880
Test set size: 220
Class distribution before resampling (train):
mental_health_history
0    446
1    434
Name: count, dtype: int64


In [11]:
from sklearn.utils import resample

# Oversample the minority class to match the majority class size
df_minority_upsampled = resample(
    df_minority,
    replace=True,                     # sample with replacement
    n_samples=len(df_majority),       # match majority class size
    random_state=42
)

# Combine majority class with upsampled minority class
df_train_balanced = pd.concat([df_majority, df_minority_upsampled])

# Shuffle the balanced dataset
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("Class distribution after resampling (train):")
print(df_train_balanced['mental_health_history'].value_counts())

# Separate features and target from the balanced training set
X_train_balanced = df_train_balanced.drop('mental_health_history', axis=1)
y_train_balanced = df_train_balanced['mental_health_history']


Class distribution after resampling (train):
mental_health_history
1    446
0    446
Name: count, dtype: int64


## Feature Extraction

In [12]:
# Your features are already numeric/categorical, no TF-IDF needed
X_train_final = X_train_balanced
X_test_final = X_test

print("Number of features:", X_train_final.shape[1])


Number of features: 20


# KNN

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize KNN (you can tune n_neighbors later)
knn = KNeighborsClassifier()

# Train on balanced dataset
knn.fit(X_train_final, y_train_balanced)

# Predict on original test set
y_pred_knn = knn.predict(X_test_final)

# Evaluation metrics
acc_knn = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

print("KNN Performance:")
print(f"Accuracy: {acc_knn:.4f}, Precision: {prec_knn:.4f}, Recall: {rec_knn:.4f}, F1-Score: {f1_knn:.4f}")


KNN Performance:
Accuracy: 0.8000, Precision: 0.8077, Recall: 0.7778, F1-Score: 0.7925


# SVM

In [15]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize SVM (linear kernel, can try 'rbf' too)
svm = SVC(kernel='linear', probability=True, random_state=42)

# Train on balanced training data
svm.fit(X_train_final, y_train_balanced)

# Predict on original test data
y_pred_svm = svm.predict(X_test_final)

# Evaluation metrics
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm)
rec_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("SVM Performance:")
print(f"Accuracy: {acc_svm:.4f}, Precision: {prec_svm:.4f}, Recall: {rec_svm:.4f}, F1-Score: {f1_svm:.4f}")


SVM Performance:
Accuracy: 0.7955, Precision: 0.7739, Recall: 0.8241, F1-Score: 0.7982


# Decision Tree

In [16]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_final, y_train_balanced)
y_pred_dt = dt.predict(X_test_final)

acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt)
rec_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

print("Decision Tree Performance:")
print(f"Accuracy: {acc_dt:.4f}, Precision: {prec_dt:.4f}, Recall: {rec_dt:.4f}, F1-Score: {f1_dt:.4f}")


Decision Tree Performance:
Accuracy: 0.7955, Precision: 0.8247, Recall: 0.7407, F1-Score: 0.7805


# Random Forest

In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_final, y_train_balanced)
y_pred_rf = rf.predict(X_test_final)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Random Forest Performance:")
print(f"Accuracy: {acc_rf:.4f}, Precision: {prec_rf:.4f}, Recall: {rec_rf:.4f}, F1-Score: {f1_rf:.4f}")

Random Forest Performance:
Accuracy: 0.7773, Precision: 0.8471, Recall: 0.6667, F1-Score: 0.7461


# AdaBoost

In [18]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=50, random_state=42)
ada.fit(X_train_final, y_train_balanced)
y_pred_ada = ada.predict(X_test_final)

acc_ada = accuracy_score(y_test, y_pred_ada)
prec_ada = precision_score(y_test, y_pred_ada)
rec_ada = recall_score(y_test, y_pred_ada)
f1_ada = f1_score(y_test, y_pred_ada)

print("AdaBoost Performance:")
print(f"Accuracy: {acc_ada:.4f}, Precision: {prec_ada:.4f}, Recall: {rec_ada:.4f}, F1-Score: {f1_ada:.4f}")


AdaBoost Performance:
Accuracy: 0.8045, Precision: 0.8095, Recall: 0.7870, F1-Score: 0.7981


# Gradiant Boosting

In [19]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_final, y_train_balanced)
y_pred_gb = gb.predict(X_test_final)

acc_gb = accuracy_score(y_test, y_pred_gb)
prec_gb = precision_score(y_test, y_pred_gb)
rec_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)

print("Gradient Boosting Performance:")
print(f"Accuracy: {acc_gb:.4f}, Precision: {prec_gb:.4f}, Recall: {rec_gb:.4f}, F1-Score: {f1_gb:.4f}")


Gradient Boosting Performance:
Accuracy: 0.8045, Precision: 0.8421, Recall: 0.7407, F1-Score: 0.7882


# XGBoost

In [20]:
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb.fit(X_train_final, y_train_balanced)
y_pred_xgb = xgb.predict(X_test_final)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print("XGBoost Performance:")
print(f"Accuracy: {acc_xgb:.4f}, Precision: {prec_xgb:.4f}, Recall: {rec_xgb:.4f}, F1-Score: {f1_xgb:.4f}")


XGBoost Performance:
Accuracy: 0.7864, Precision: 0.8506, Recall: 0.6852, F1-Score: 0.7590


# LightGBM

In [21]:
    from lightgbm import LGBMClassifier

    lgbm = LGBMClassifier(random_state=42)
    lgbm.fit(X_train_final, y_train_balanced)
    y_pred_lgbm = lgbm.predict(X_test_final)

    acc_lgbm = accuracy_score(y_test, y_pred_lgbm)
    prec_lgbm = precision_score(y_test, y_pred_lgbm)
    rec_lgbm = recall_score(y_test, y_pred_lgbm)
    f1_lgbm = f1_score(y_test, y_pred_lgbm)

    print("LightGBM Performance:")
    print(f"Accuracy: {acc_lgbm:.4f}, Precision: {prec_lgbm:.4f}, Recall: {rec_lgbm:.4f}, F1-Score: {f1_lgbm:.4f}")


[LightGBM] [Info] Number of positive: 446, number of negative: 446
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000251 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 176
[LightGBM] [Info] Number of data points in the train set: 892, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g